In [2]:
!pip -q install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.3 MB/s eta 0:00:00:00:0100:01


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

set_seed(42)

### Load raw dataset splits

In [4]:
sft_train_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_sft")
sft_val_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_sft")

dpo_train_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
dpo_val_raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_prefs")


print("Raw SFT Train:", len(sft_train_raw))
print("Raw SFT Val  :", len(sft_val_raw))
print("Raw DPO Train:", len(dpo_train_raw))
print("Raw DPO Val  :", len(dpo_val_raw))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train_prefs-00000-of-00001.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/test_prefs-00000-of-00001.parquet:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

...

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Raw SFT Train: 61135
Raw SFT Val  : 1000
Raw DPO Train: 61135
Raw DPO Val  : 2000


_Make smaller splits and helper function_

In [6]:
sft_train_raw = sft_train_raw.shuffle(seed=42).select(range(3000))
sft_val_raw = sft_val_raw.select(range(200))

dpo_train_raw = dpo_train_raw.shuffle(seed=42).select(range(3000))
dpo_val_raw = dpo_val_raw.select(range(200))

print("SFT Train:", len(sft_train_raw))
print("SFT Val  :", len(sft_val_raw))
print("DPO Train:", len(dpo_train_raw))
print("DPO Val  :", len(dpo_val_raw))

SFT Train: 3000
SFT Val  : 200
DPO Train: 3000
DPO Val  : 200


_Inspect one raw SFT sample and one raw DPO sample_

In [7]:
def get_last_assistant_text(messages):
    for message in reversed(messages):
        if message["role"] == "assistant":
            return message["content"].strip()
    return ""


sft_sample = sft_train_raw[0]
dpo_sample = dpo_train_raw[0]

print("Raw SFT Prompt:\n")
print(sft_sample["prompt"])
print("\nRaw SFT Chosen Answer:\n")
print(get_last_assistant_text(sft_sample["chosen"]))

print("\n" + "=" * 80 + "\n")

print("Raw DPO Prompt:\n")
print(dpo_sample["prompt"])
print("\nRaw DPO Chosen:\n")
print(get_last_assistant_text(dpo_sample["chosen"]))
print("\nRaw DPO Rejected:\n")
print(get_last_assistant_text(dpo_sample["rejected"]))

Raw SFT Prompt:

Do you know something about crystallography and structure factor?

Raw SFT Chosen Answer:

Crystallography is the science of the arrangement of atoms in solids. It is a vast and interdisciplinary field that has applications in physics, chemistry, materials science, biology, and engineering.

The structure factor is a mathematical function that is used to describe the diffraction of waves by a crystal. It is a complex number that is related to the atomic positions in the crystal.

The structure factor can be used to calculate the intensity of the diffracted waves. This information can be used to determine the atomic positions in the crystal and to study the structure of materials.

Crystallography is a powerful tool for understanding the structure of materials. It has been used to determine the structures of many important materials, including metals, semiconductors, and pharmaceuticals. It is also used to study the structure of biological materials, such as proteins an